In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / 'dataset').exists() and (candidate / 'paths.py').exists():
        candidate_str = str(candidate.resolve())
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)
        break


### Import packages and load data

In [ ]:
from pathlib import Path

import pandas as pd
from qdrant_client import QdrantClient
from transformers import AutoModel

from config import (
    QDRANT_ICD_PROCEDURE_COLLECTION_NAME,
    QDRANT_ICD_PROCEDURE_EMBEDDING_MODEL,
    QDRANT_URL,
)
from paths import QDRANT_STORAGE_DIR
from qdrant_collection import Qdrant_Collection, batch_upsert_procedures

from dataset.data import BASE_HOSP

d_icd_procedures_path = "d_icd_procedures.csv"
procedures_path = "procedures_icd.csv"

d_icd_procedures = pd.read_csv(BASE_HOSP.joinpath(d_icd_procedures_path))
procedures = pd.read_csv(BASE_HOSP.joinpath(procedures_path))

### Start the docker client

From `HospitalAgent/`, either start `qdrant/qdrant` via the Docker GUI or run

```
docker run -p 6333:6333 -p 6334:6334 \
  -v "$(pwd)/src/raw/runtime/qdrant/main:/qdrant/storage:z" \
  -e QDRANT__TELEMETRY_DISABLED=true \
  qdrant/qdrant
```
in the terminal (same canonical storage location used by runtime procedure lookup).

For more info, please check out: https://qdrant.tech/documentation/quickstart/.

# Run embedding

- Load Huggingface embedding model
- create feature vectors
- upsert into Qdrant DB

In [ ]:
# skip this cell
# embedding_client = AutoModel.from_pretrained(
#     QDRANT_ICD_PROCEDURE_EMBEDDING_MODEL, trust_remote_code=True
# )

# qdrant_client = QdrantClient(url=QDRANT_URL)
# collection = Qdrant_Collection(
#     qdrant_client,
#     embedding_client,
#     QDRANT_ICD_PROCEDURE_COLLECTION_NAME,
#     QDRANT_ICD_PROCEDURE_EMBEDDING_MODEL,
# )

# batch_upsert_procedures(
#     collection, d_icd_procedures.to_dict(orient="records"), max_batch_size=200
# )

# Run embedding with persistent local qdrant storage

In [ ]:
embedding_client = AutoModel.from_pretrained(
    QDRANT_ICD_PROCEDURE_EMBEDDING_MODEL, trust_remote_code=True
)

qdrant_client = QdrantClient(path=str(QDRANT_STORAGE_DIR))
collection = Qdrant_Collection(
    qdrant_client,
    embedding_client,
    QDRANT_ICD_PROCEDURE_COLLECTION_NAME,
    QDRANT_ICD_PROCEDURE_EMBEDDING_MODEL,
)

batch_upsert_procedures(
    collection, d_icd_procedures.to_dict(orient="records"), max_batch_size=200
)

In [ ]:
collection.search("appendectomy", query_filter=None, top_k=10)